# Notebook 1: SAM 3 segmentation and weight estimation (no training)

This notebook uses **SAM 3 alone** to:

1. segment the **broiler**, the **scale** and the **feeder (feeding pan)** in every labelled frame of your FlockLabel validation export;
2. measure the feeder (known diameter **35.56 cm = 14 in**) to get the **pixel-to-cm** scale;
3. pick the bird that the scale is weighing;
4. rotate its mask to the body axis, cut it into **slices (cylinders)**, add up their volumes, and multiply by **density** to get its weight;
5. compare the estimate with the scale reading and search for the best **density x slice count**.

```
frame -> SAM 3 (text prompts) -> feeder mask -> pixel_to_cm
                               -> broiler masks -> bird on scale -> align -> slices -> volume x density -> kg
```

**Install once** (in a terminal, from the package folder):

```bash
pip install -e ".[sam,notebook]"
hf auth login        # SAM 3 weights are gated: request access at huggingface.co/facebook/sam3 first
```

On an Apple-silicon MacBook, SAM 3 runs through Hugging Face `transformers` on the **MPS** GPU, so expect a few seconds per frame. Meta's own SAM 3 code needs CUDA, which is why this package doesn't use it.

> **Tip:** set `DRY_RUN = True` below to try the whole notebook on generated demo data, with no SAM 3 and no GPU.

## 0. Settings

In [1]:
# ---- change these -------------------------------------------------------------
DRY_RUN = False                 # True = synthetic demo data + colour segmenter (no SAM 3)
MAX_FRAMES_PER_EVENT = 10       # labelled frames used per weighing event (None = all)
DENSITY = 900.0                 # kg/m^3, repository default
SLICE_COUNT = None              # None = one slice per pixel column (repository default)
CALIBRATION_METHOD = "enclosing_circle"   # or "ellipse_major", "equivalent_area"
CALIBRATION_MODE = "video_median"         # or "frame", "export_hint"
DENSITY_GRID = list(range(600, 1301, 25)) # densities to sweep (kg/m^3)
SLICE_GRID = [None, 5, 8, 10, 12, 15, 20, 25, 30, 40]  # slice counts to sweep
CALIBRATION_FRACTION = 0.5      # share of events used to fit density (rest = held out)
SEED = 0
SAVE_FIGURES = True
OUTPUT_DIR = "outputs/01_sam3_weight"
# -------------------------------------------------------------------------------

In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import broilerseg
from broilerseg import viz
from broilerseg.calibration import calibrate_from_hsv, calibrate_from_mask, summarize_calibrations
from broilerseg.config import PromptConfig, VolumeConfig
from broilerseg.evaluation import (best_density, event_table, regression_metrics, split_events,
                                   sweep_grid)
from broilerseg.experiments import WeightExperiment
from broilerseg.io import load_flocklabel_export, subsample_per_event
from broilerseg.notebook import ask_for_directory, environment_report
from broilerseg.segmentation import clean_mask, select_bird
from broilerseg.volume import estimate_volume, volume_by_slice_count

warnings.filterwarnings("ignore", category=UserWarning)
viz.set_style()
pd.set_option("display.max_columns", 40)
OUT = Path(OUTPUT_DIR); (OUT / "figures").mkdir(parents=True, exist_ok=True)
FIG = 0
def show(fig, name):
    # Display a figure and (optionally) save it as a numbered PNG.
    global FIG
    FIG += 1
    if SAVE_FIGURES:
        viz.save(fig, OUT / "figures" / f"{FIG:02d}_{name}.png")
    plt.show()

print("broilerseg", broilerseg.__version__)
display(pd.Series(environment_report(), name="environment").to_frame())

OSError: [Errno 30] Read-only file system: 'outputs'

## 1. Point the notebook at your FlockLabel export

Paste the folder that FlockLabel's **Export for evaluation** created: it holds `test_videos/` and `ground_truth/`. A FlockLabel **dataset** export (with `manifest.csv`) also works. You can paste a folder up to two levels above it.

To skip this prompt on later runs, set the environment variable `BROILERSEG_WEIGHT_DIR`.

In [ ]:
if DRY_RUN:
    from broilerseg.synthetic import make_demo_data
    demo = make_demo_data(OUT / "demo_data")
    WEIGHT_DIR = demo["export"]
    print("DRY RUN: using synthetic export at", WEIGHT_DIR)
else:
    WEIGHT_DIR = ask_for_directory("Paste the FlockLabel export folder (weight validation data)",
                                   env_var="BROILERSEG_WEIGHT_DIR")

export = load_flocklabel_export(WEIGHT_DIR)
print(export)
print("Export info:", export.info)
display(export.events.head(10))
display(export.samples.head(10))

### 1a. What's in the validation data?
Each dot is the scale reading on one labelled frame. The orange bar is the event median, which is the value the event-level metrics use.

In [ ]:
show(viz.plot_weight_dataset_overview(export.samples), "dataset_overview")
samples = subsample_per_event(export.samples, MAX_FRAMES_PER_EVENT, SEED)
print(f"Using {len(samples)} of {len(export.samples)} labelled frames "
      f"({samples.event_id.nunique()} events, <= {MAX_FRAMES_PER_EVENT} frames each)")

## 2. Load SAM 3

SAM 3 segments a *concept* named by a short phrase. Each class has a few phrases; detections from every phrase are pooled, and duplicates are removed with mask non-maximum suppression. If a class is missed on your footage, edit the phrases here and re-run.

In [ ]:
prompts = PromptConfig()
# Example edits:
# prompts.prompts["feeder"] = ["orange feeder pan", "round orange tray"]
# prompts.score_threshold = 0.35
display(pd.DataFrame([(c, p) for c, ps in prompts.prompts.items() for p in ps],
                     columns=["class", "prompt"]))

if DRY_RUN:
    from broilerseg.synthetic import ColorSegmenter
    segmenter = ColorSegmenter()
else:
    from broilerseg.segmentation import Sam3Segmenter
    segmenter = Sam3Segmenter(prompts=prompts, max_side=1280)   # device picked automatically
    print("SAM 3 running on:", segmenter.device)

## 3. One frame, step by step

Before running everything, follow one labelled frame through the whole pipeline. Change `EXAMPLE` to look at another one.

In [ ]:
EXAMPLE = 0
exp_one = WeightExperiment(segmenter, VolumeConfig(DENSITY, SLICE_COUNT),
                           CALIBRATION_METHOD, "frame")
rec_one = exp_one.run(samples.iloc[[EXAMPLE]])
sid = int(rec_one.sample_id.iloc[0])
s = samples.iloc[EXAMPLE]
image = exp_one.image(sid)
result = exp_one.results[sid]
print(f"{s.video_name} frame {s.frame_label}: scale reads {s.weight_kg:.3f} kg | "
      f"SAM 3 took {result.seconds:.2f} s | found {result.counts()}")

### 3a. What SAM 3 found: one panel per class, the overlay and the final label map

In [ ]:
show(viz.plot_segmentation_panels(image, result, f"{s.video_name}  frame {s.frame_label}"), "example_panels")
show(viz.plot_overlay(image, result, title="All instances with confidence"), "example_overlay")
show(viz.plot_instances_by_prompt(image, result), "example_prompts")

### 3b. Pixel-to-cm from the feeder
The feeder's convex hull handles a bird hiding part of the rim. Three diameter measurements are compared, and the cyan one is used. The repository's colour-based (HSV) detector runs as a cross-check.

In [ ]:
feeder = result.best("feeder")
cal = calibrate_from_mask(feeder.mask if feeder else None, method=CALIBRATION_METHOD)
show(viz.plot_feeder_calibration(image, cal, feeder.mask if feeder else None), "example_calibration")
import cv2
hsv_cal = calibrate_from_hsv(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
print(f"SAM 3 feeder : {cal.pixel_to_cm:.5f} cm/px")
print(f"HSV (repo)   : {hsv_cal.pixel_to_cm:.5f} cm/px")
print(f"FlockLabel   : {s.pixel_to_cm_hint} (from events.csv, may be empty)")

### 3c. Which bird is the scale weighing?
The broiler whose box best matches the FlockLabel box is chosen. Without a box, the bird with the most of its mask over the scale and feeder platform is chosen. Then the mask is cleaned: the largest blob is kept and holes are filled.

In [ ]:
bird, info = select_bird(result, (s.x1, s.y1, s.x2, s.y2))
show(viz.plot_bird_selection(image, result, bird, (s.x1, s.y1, s.x2, s.y2), info), "example_selection")
if bird is not None:
    show(viz.plot_mask_cleanup(image, bird.mask, clean_mask(bird.mask)), "example_cleanup")

### 3d. From mask to volume to weight
The mask is rotated so the body axis is horizontal, then split into slices. Each slice is treated as a cylinder whose diameter is the mask's height at the slice centre:

$V = \pi \sum_i r_i^2\, w_i$ (cm³), and $\text{mass} = V \cdot 10^{-6} \cdot \rho$ (kg).

In [ ]:
if sid in exp_one.estimates:
    bird_c = exp_one.birds[sid]
    est = exp_one.estimates[sid]
    show(viz.plot_alignment(bird_c.mask, est), "example_alignment")
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, k in zip(axes, [None, 20, 8]):
        viz.plot_slices(estimate_volume(bird_c.mask, est.pixel_to_cm, DENSITY, k), ax=ax)
    fig.suptitle("Same bird, three slice counts", fontweight="bold"); fig.tight_layout()
    show(fig, "example_slice_counts")
    show(viz.plot_slice_profile(est), "example_profile")
    show(viz.plot_cylinders_3d(estimate_volume(bird_c.mask, est.pixel_to_cm, DENSITY, 25)), "example_3d")
    show(viz.plot_slice_convergence(volume_by_slice_count(bird_c.mask, est.pixel_to_cm, SLICE_GRID)),
         "example_convergence")
    show(viz.plot_volume_storyboard(image, result, bird_c, est, s.weight_kg,
                                    f"{s.video_name} frame {s.frame_label}"), "example_storyboard")
else:
    print("No estimate for this frame:", rec_one.status.iloc[0], "- try another EXAMPLE")

## 4. Run on every selected frame
SAM 3 runs once per frame, and the results are cached in `experiment.results`, so re-running the analysis cells below is instant.

In [ ]:
experiment = WeightExperiment(segmenter, VolumeConfig(DENSITY, SLICE_COUNT),
                              CALIBRATION_METHOD, CALIBRATION_MODE)
records = experiment.run(samples)
records.to_csv(OUT / "records_sam3.csv", index=False)
print(f"{records.pred_kg.notna().sum()} / {len(records)} frames have an estimate")
display(records.head(10))
show(viz.plot_status_breakdown(records), "status")

### 4a. Segmentation across the dataset

In [ ]:
rng = np.random.default_rng(SEED)
pick = rng.choice(records.sample_id, size=min(12, len(records)), replace=False)
items = [(experiment.image(int(i)), experiment.results[int(i)],
          f"{records.set_index('sample_id').loc[i, 'video_name'][:18]} f{records.set_index('sample_id').loc[i, 'frame_label']}")
         for i in pick]
show(viz.plot_gallery(items, ncols=4, title="Random frames: SAM 3 masks"), "gallery")
show(viz.plot_score_distributions(list(experiment.results.values())), "scores")
cnt = pd.DataFrame([r.counts() for r in experiment.results.values()])
fig, ax = plt.subplots(figsize=(8, 3.5))
cnt.apply(lambda c: c.value_counts()).fillna(0).T.plot.bar(stacked=True, ax=ax, colormap="viridis")
ax.set_ylabel("frames"); ax.set_title("How many instances per class per frame"); ax.legend(title="count")
show(fig, "instance_counts")

### 4b. Is the scale stable within each video?
The camera is fixed, so every frame of a video should give the same cm/px. A high CV% means the feeder masks are unreliable for that video.

In [ ]:
show(viz.plot_calibration_stability(records), "calibration_stability")
display(summarize_calibrations(records, value="pixel_to_cm_frame"))
show(viz.plot_calibration_methods([experiment.calibrations[i] for i in records.sample_id]),
     "calibration_methods")

## 5. How accurate is the weight? (density = setting above)

In [ ]:
print("Frame level :", {k: round(v, 3) for k, v in regression_metrics(records.actual_kg, records.pred_kg).items()})
ev = event_table(records)
print("Event level :", {k: round(v, 3) for k, v in regression_metrics(ev.actual_kg, ev.pred_kg).items()})
display(ev.round(3))
show(viz.plot_accuracy_overview(records, f"SAM 3 masks, density {DENSITY:.0f}, slices {SLICE_COUNT}"), "accuracy")
show(viz.plot_per_event(records), "per_event")
show(viz.plot_frame_traces(records), "frame_traces")
show(viz.plot_error_analysis(records), "error_analysis")

### 5a. What drives the error?
*Implied density* is the density that would make a frame exactly right. A tight distribution means one density fits all birds. A trend against bird size means the cylinder model is biased by body shape or posture.

In [ ]:
show(viz.plot_implied_density(records, DENSITY), "implied_density")
show(viz.plot_feature_relationships(records), "feature_relationships")

### 5b. Best and worst frames, end to end

In [ ]:
ok = records.dropna(subset=["pred_kg"]).assign(abs_pct=lambda d: d.pct_error.abs()).sort_values("abs_pct")
for tag, rows in (("BEST", ok.head(2)), ("WORST", ok.tail(3))):
    for _, r in rows.iterrows():
        i = int(r.sample_id)
        show(viz.plot_volume_storyboard(experiment.image(i), experiment.results[i], experiment.birds[i],
                                        experiment.estimates[i], r.actual_kg,
                                        f"{tag}: {r.video_name} frame {r.frame_label} ({r.pct_error:+.1f}%)"),
             f"storyboard_{tag.lower()}_{i}")

## 6. Search density x slice count
Events are split into a **calibration** set, used to choose parameters, and a **held-out** set, used to report error. The volume doesn't depend on density, so the volumes are computed once per slice count and every density is scored from those.

In [ ]:
cal_events, held_events = split_events(records.dropna(subset=["pred_kg"]).event_id, CALIBRATION_FRACTION, SEED)
print(f"calibration events: {cal_events}\nheld-out events  : {held_events}")
vol_table = experiment.volume_table(SLICE_GRID)
grid = sweep_grid(vol_table, DENSITY_GRID, cal_events, held_events)
grid.to_csv(OUT / "density_slice_grid.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(22, 5))
viz.plot_sweep_heatmap(grid, "cal_rmse_kg", ax=axes[0])
viz.plot_sweep_heatmap(grid, "held_rmse_kg", ax=axes[1])
fig.suptitle("Event-level RMSE (kg): calibration (left) vs held-out (right)", fontweight="bold")
show(fig, "sweep_heatmaps")
show(viz.plot_sweep_curves(grid, "held_rmse_kg"), "sweep_curves")
bd = best_density(vol_table, cal_events, held_events)
display(bd.round(4))

In [ ]:
# Re-score with the best (slice count, density) chosen on CALIBRATION events only
best = grid.sort_values("cal_rmse_kg").iloc[0]
best_slices = None if best.slice_count == "per-column" else int(best.slice_count)
print(f"Chosen on calibration events: slices={best.slice_count}, density={best.density_kg_m3:.0f} kg/m^3")
tuned = WeightExperiment(segmenter, VolumeConfig(best.density_kg_m3, best_slices),
                         CALIBRATION_METHOD, CALIBRATION_MODE, results=experiment.results)
rec_tuned = tuned.run(samples)
rec_tuned.to_csv(OUT / "records_sam3_tuned.csv", index=False)
held = rec_tuned[rec_tuned.event_id.isin(held_events)]
show(viz.plot_accuracy_overview(held, "Held-out events with tuned parameters"), "accuracy_tuned_heldout")
show(viz.plot_method_comparison({f"default ({DENSITY:.0f})": records[records.event_id.isin(held_events)],
                                 f"tuned ({best.density_kg_m3:.0f}, {best.slice_count})": held},
                                "Default vs tuned parameters (held-out events)"), "default_vs_tuned")

## 7. Summary

In [ ]:
summary = pd.DataFrame({
    "default": regression_metrics(event_table(records).actual_kg, event_table(records).pred_kg),
    "tuned (all events)": regression_metrics(event_table(rec_tuned).actual_kg, event_table(rec_tuned).pred_kg),
    "tuned (held-out only)": regression_metrics(event_table(held).actual_kg, event_table(held).pred_kg),
}).T.round(3)
display(summary)
summary.to_csv(OUT / "summary.csv")
print(f"Figures and tables saved in {OUT.resolve()}")

**Reading the results**

* If the **held-out** RMSE is much worse than the **calibration** RMSE, the chosen density is over-fitted. Collect more events.
* If the implied-density plot has a **slope against size**, a single density can't fit every bird. Try another slice count or a size-dependent correction.
* Use the frames flagged in *status* and the **worst** storyboards to find segmentation failures. These are the cases the YOLO student (Notebook 2) must handle.